In [1]:
import os
import sys
import ctypes
from pathlib import Path
import numpy as np
import pandas as pd
import napari
from tqdm.auto import tqdm
import matplotlib.colors as mcolors
import colorsys
# Make NVRTC/NVIDIA DLLs discoverable before importing CuPy.
if os.name == 'nt':
    torch_lib = Path(sys.prefix) / 'Lib' / 'site-packages' / 'torch' / 'lib'
    dll_dirs = [
        Path(sys.prefix) / 'Library' / 'bin',
        torch_lib,
    ]
    for dll_dir in dll_dirs:
        if dll_dir.exists():
            os.add_dll_directory(str(dll_dir))

    if torch_lib.exists():
        os.environ['PATH'] = f"{torch_lib};" + os.environ.get('PATH', '')
        for name in ('nvrtc64_120_0.dll', 'nvrtc-builtins64_128.dll'):
            dll_path = torch_lib / name
            if dll_path.exists():
                ctypes.WinDLL(str(dll_path))

import cupy as cp
import cupyx.scipy.ndimage as ndi_gpu
import dask.array as da
from skimage.morphology import skeletonize as skeletonize_3d
import sknw
from scipy.spatial.distance import cdist
from scipy.ndimage import distance_transform_edt as edt
from scipy import ndimage as ndi

c:\Users\taylorhearn\AppData\Local\miniconda3\envs\clean_vascumap\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def custom_napari_palette(n_labels, base_colour, hue_span=0.02, sat_span=0.45, light_span=0.45, seed=0, alpha=1.0):
    """Build a varied Napari label colormap by jittering H/S/L around a base Matplotlib color."""
    n_labels = int(n_labels)
    if n_labels < 1:
        return {0: np.array([0.0, 0.0, 0.0, 0.0], dtype=float)}

    rng = np.random.default_rng(seed)
    r, g, b = mcolors.to_rgb(base_colour)
    base_h, base_l, base_s = colorsys.rgb_to_hls(r, g, b)

    hue = (base_h + rng.uniform(-hue_span, hue_span, n_labels)) % 1.0
    sat = np.clip(base_s + rng.uniform(-sat_span, sat_span, n_labels), 0.20, 1.00)
    light = np.clip(base_l + rng.uniform(-light_span, light_span, n_labels), 0.18, 0.88)

    order = rng.permutation(n_labels)
    custom_colormap = {0: np.array([0.0, 0.0, 0.0, 0.0], dtype=float)}
    for idx, j in enumerate(order, start=1):
        pr, pg, pb = colorsys.hls_to_rgb(float(hue[j]), float(light[j]), float(sat[j]))
        custom_colormap[idx] = np.array([pr, pg, pb, float(alpha)], dtype=float)
    return custom_colormap

In [3]:
def safe_divide(numerator, denominator):
    denominator = float(denominator)
    if denominator <= 0:
        return np.nan
    return float(numerator) / denominator

def safe_median(values):
    arr = np.asarray(pd.to_numeric(values, errors='coerce'), dtype=float).ravel()
    arr = arr[~np.isnan(arr)]
    if arr.size == 0:
        return np.nan
    return float(np.median(arr))

def safe_percentile_spread(values, low=10, high=90):
    arr = np.asarray(pd.to_numeric(values, errors='coerce'), dtype=float).ravel()
    arr = arr[~np.isnan(arr)]
    if arr.size == 0:
        return np.nan
    q = np.percentile(arr, [low, high])
    return float(q[1] - q[0])

def cupy_chunk_processing(volume, processing_func, chunk_size=(64, 512, 512), overlap=(15, 15, 15), *args, **kwargs):
    result = np.empty_like(volume)
    pool = cp.get_default_memory_pool()
    z_steps = range(0, volume.shape[0], chunk_size[0])

    for z in tqdm(z_steps, desc='Processing chunks'):
        for y in range(0, volume.shape[1], chunk_size[1]):
            for x in range(0, volume.shape[2], chunk_size[2]):
                z_start = max(0, z - overlap[0])
                z_end = min(volume.shape[0], z + chunk_size[0] + overlap[0])
                y_start = max(0, y - overlap[1])
                y_end = min(volume.shape[1], y + chunk_size[1] + overlap[1])
                x_start = max(0, x - overlap[2])
                x_end = min(volume.shape[2], x + chunk_size[2] + overlap[2])

                chunk = volume[z_start:z_end, y_start:y_end, x_start:x_end]
                chunk_gpu = cp.asarray(chunk)

                def func_wrapper(gpu_chunk):
                    gpu_args = []
                    for arg in args:
                        gpu_args.append(cp.asarray(arg) if isinstance(arg, np.ndarray) else arg)

                    gpu_kwargs = {}
                    for k, v in kwargs.items():
                        gpu_kwargs[k] = cp.asarray(v) if isinstance(v, np.ndarray) else v

                    return processing_func(gpu_chunk, *gpu_args, **gpu_kwargs)

                filtered_chunk = func_wrapper(chunk_gpu)

                w_z_start = z - z_start
                w_z_end = w_z_start + chunk_size[0]
                w_y_start = y - y_start
                w_y_end = w_y_start + chunk_size[1]
                w_x_start = x - x_start
                w_x_end = w_x_start + chunk_size[2]

                valid_chunk = filtered_chunk[
                    w_z_start:min(w_z_end, filtered_chunk.shape[0]),
                    w_y_start:min(w_y_end, filtered_chunk.shape[1]),
                    w_x_start:min(w_x_end, filtered_chunk.shape[2]),
                ].get()

                result_z = min(z, result.shape[0])
                result_y = min(y, result.shape[1])
                result_x = min(x, result.shape[2])

                result[
                    result_z:result_z + valid_chunk.shape[0],
                    result_y:result_y + valid_chunk.shape[1],
                    result_x:result_x + valid_chunk.shape[2],
                ] = valid_chunk

                del chunk_gpu, filtered_chunk
                pool.free_all_blocks()

    return result

def measure_edge_length(coordinates):
    differences = np.diff(coordinates, axis=0)
    segment_lengths = np.linalg.norm(differences, axis=1)
    return np.sum(segment_lengths)

def prune_graph(graph, area_3d, edt_cutoff=0.25, length_cutoff=25):
    while True:
        endpoint_nodes = [node for node, degree in graph.degree() if degree == 1]
        values = []
        for node in endpoint_nodes:
            neighbors = list(graph.neighbors(node))

            if len(neighbors) == 1:
                neighbor = neighbors[0]
                edge_data = graph.get_edge_data(neighbor, node)
                edge_length = measure_edge_length(edge_data['pts'])
                branch_edt = area_3d[edge_data['pts'][:, 0], edge_data['pts'][:, 1], edge_data['pts'][:, 2]]
                branch_edt_interp = np.interp(np.linspace(0, 1, 100), np.linspace(0, 1, branch_edt.size), branch_edt)

                if np.mean(branch_edt_interp[:50]) < np.mean(branch_edt_interp[-50:]):
                    neighbor_coords = edge_data['pts'][-1]
                    part_oi = branch_edt_interp[:20]
                else:
                    neighbor_coords = edge_data['pts'][0]
                    part_oi = branch_edt_interp[-20:]

                neighbor_edt = area_3d[neighbor_coords[0], neighbor_coords[1], neighbor_coords[2]]
                value = np.mean(part_oi) / (neighbor_edt + 1e-6)

                if value > edt_cutoff or edge_length <= length_cutoff:
                    graph.remove_node(node)
                    values.append(value)

        if len(values) == 0:
            break
    return graph

def remove_mid_node(graph):
    while True:
        nodes_to_process = [n for n, d in graph.degree() if d == 2]
        if not nodes_to_process:
            break

        processed_in_iteration = False
        for i in nodes_to_process:
            if not graph.has_node(i) or graph.degree(i) != 2:
                continue

            neighbors = list(graph.neighbors(i))
            if len(neighbors) != 2:
                continue

            n1, n2 = neighbors[0], neighbors[1]
            if n1 == n2 or graph.has_edge(n1, n2):
                continue

            edge1 = graph.get_edge_data(i, n1)
            edge2 = graph.get_edge_data(i, n2)
            pts1 = np.atleast_2d(edge1['pts'])
            pts2 = np.atleast_2d(edge2['pts'])
            node_coord = graph.nodes[i]['pts'].astype(np.int32)

            s1, e1 = pts1[0], pts1[-1]
            s2, e2 = pts2[0], pts2[-1]

            dists = cdist([s1, e1], [s2, e2])
            min_row, min_col = np.unravel_index(np.argmin(dists), dists.shape)

            if min_row == 0 and min_col == 0:
                combined_line = np.concatenate([pts1[::-1], [node_coord], pts2], axis=0)
            elif min_row == 1 and min_col == 1:
                combined_line = np.concatenate([pts1, [node_coord], pts2[::-1]], axis=0)
            elif min_row == 0 and min_col == 1:
                combined_line = np.concatenate([pts2[::-1], [node_coord], pts1], axis=0)
            else:
                combined_line = np.concatenate([pts1, [node_coord], pts2], axis=0)

            new_weight = edge1.get('weight', 0) + edge2.get('weight', 0)
            graph.add_edge(n1, n2, weight=new_weight, pts=combined_line)
            graph.remove_node(i)
            processed_in_iteration = True

        if not processed_in_iteration:
            break
    return graph

def collect_border_vicinity_edges(graph, image_shape, vicinity_xy=50):
    border_vicinity_edges = set()
    for u, v in graph.edges():
        try:
            pts = graph[u][v]['pts']
            if any((
                pt[1] < vicinity_xy or pt[1] > image_shape[1] - 1 - vicinity_xy or
                pt[2] < vicinity_xy or pt[2] > image_shape[2] - 1 - vicinity_xy
                ) for pt in pts):
                border_vicinity_edges.add((u, v))
        except KeyError:
            continue

    trimmed_subgraph = graph.copy()
    edges_to_remove = [edge for edge in border_vicinity_edges if trimmed_subgraph.has_edge(*edge)]
    trimmed_subgraph.remove_edges_from(edges_to_remove)

    isolated_nodes = [node for node in trimmed_subgraph.nodes() if trimmed_subgraph.degree[node] == 0]
    if isolated_nodes:
        trimmed_subgraph.remove_nodes_from(isolated_nodes)

    return trimmed_subgraph

def compute_cross_sectional_areas(mask, skeleton, binary_edt, voxel_size_um=(2.0, 2.0, 2.0)):
    voxel_size_um = np.asarray(voxel_size_um, dtype=float)
    edt_2d = edt(np.max(mask, axis=0), sampling=tuple(voxel_size_um[1:]))
    area_3d = np.zeros_like(binary_edt, dtype=float)
    z_idx, y_idx, x_idx = np.where(skeleton > 0)

    minor_axis = 2 * binary_edt[z_idx, y_idx, x_idx]
    major_axis = 2 * edt_2d[y_idx, x_idx]

    areas = np.pi * (major_axis / 2) * (minor_axis / 2)
    area_3d[z_idx, y_idx, x_idx] = areas
    return area_3d

def fractal_dimension_and_lacunarity(binary, min_box_size=1, max_box_size=None, n_samples=12):
    pts = np.argwhere(binary > 0)
    if pts.size == 0:
        return np.nan, np.nan

    if max_box_size is None:
        max_box_size = int(np.floor(np.log2(np.min(binary.shape))))

    scales = np.floor(np.logspace(max_box_size, min_box_size, num=n_samples, base=2)).astype(np.int64)
    scales = np.unique(scales)
    scales = scales[scales > 0]

    log_inv_scale = []
    log_N = []
    lac_vals = []

    for s in scales:
        box_ids = pts // s
        unique_box_ids, counts = np.unique(box_ids, axis=0, return_counts=True)  # occupied boxes + masses
        N = unique_box_ids.shape[0]
        if N < 2:
            continue

        log_inv_scale.append(np.log(1.0 / s))
        log_N.append(np.log(N))

        mu = counts.mean()
        lac_vals.append((counts.var() / (mu * mu)) if mu > 0 else np.nan)

    if len(log_N) < 2:
        return np.nan, np.nan

    fd = float(np.polyfit(log_inv_scale, log_N, 1)[0])
    lac = float(np.nanmean(lac_vals)) if len(lac_vals) else np.nan
    return fd, lac
def graph2image(graph, shape):
    pruned_skeleton = np.zeros(shape)
    for u, v in graph.edges():
        coords = graph.get_edge_data(u, v)['pts']

        clipped_coords = np.zeros_like(coords)
        clipped_coords[:, 0] = np.clip(coords[:, 0], 0, shape[0] - 1)
        clipped_coords[:, 1] = np.clip(coords[:, 1], 0, shape[1] - 1)
        clipped_coords[:, 2] = np.clip(coords[:, 2], 0, shape[2] - 1)

        pruned_skeleton[clipped_coords[:, 0], clipped_coords[:, 1], clipped_coords[:, 2]] = 1

    return pruned_skeleton

def summarize_network_headline_metrics(graph, area_image, voxel_size_um=(2.0, 2.0, 2.0), distance_mode='skeleton'):
    voxel_size_um = np.asarray(voxel_size_um, dtype=float)
    summary = {
        'median_sprout_and_branch_tortuosity': np.nan,
        'p90_minus_p10_sprout_and_branch_tortuosity': np.nan,
        'median_sprout_and_branch_median_cs_area_um2': np.nan,
        'p90_minus_p10_sprout_and_branch_median_cs_area_um2': np.nan,
        'median_junction_dist_nearest_junction_um': np.nan,
        'p90_minus_p10_junction_dist_nearest_junction_um': np.nan,
        'median_sprout_dist_nearest_endpoint_um': np.nan,
        'p90_minus_p10_sprout_dist_nearest_endpoint_um': np.nan,
    }

    tortuosities = []
    median_cs_areas = []
    for u, v in graph.edges():
        try:
            pts = graph[u][v]['pts']
            if len(pts) < 2:
                continue
            pts_um = np.asarray(pts, dtype=float) * voxel_size_um[None, :]
            segment_lengths_um = np.linalg.norm(np.diff(pts_um, axis=0), axis=1)
            length_um = float(np.sum(segment_lengths_um))
            shortest_path_um = float(np.linalg.norm(pts_um[0] - pts_um[-1]))
            tortuosities.append(np.clip(length_um / (shortest_path_um + 1e-8), 0, 5))

            segment_areas = area_image[pts[:, 0], pts[:, 1], pts[:, 2]]
            median_cs_areas.append(float(np.nanmedian(segment_areas)))
        except (KeyError, IndexError):
            continue

    if len(tortuosities) > 0:
        summary['median_sprout_and_branch_tortuosity'] = safe_median(tortuosities)
        summary['p90_minus_p10_sprout_and_branch_tortuosity'] = safe_percentile_spread(tortuosities)
    if len(median_cs_areas) > 0:
        summary['median_sprout_and_branch_median_cs_area_um2'] = safe_median(median_cs_areas)
        summary['p90_minus_p10_sprout_and_branch_median_cs_area_um2'] = safe_percentile_spread(median_cs_areas)

    nodes = list(graph.nodes())
    if len(nodes) < 2:
        return summary

    positions = np.array([graph.nodes[n]['pts'] for n in nodes])
    positions_um = positions.astype(float) * voxel_size_um[None, :]
    node_type = np.array(['sprout' if graph.nodes[n]['sprout'] else 'junction' for n in nodes])

    if distance_mode == 'skeleton':
        import networkx as nx
        graph_weighted = graph.copy()
        for u, v, data in graph_weighted.edges(data=True):
            pts = data.get('pts', None)
            if pts is None or len(pts) < 2:
                graph_weighted.edges[u, v]['path_length_um'] = np.inf
                continue
            pts_um = np.asarray(pts, dtype=float) * voxel_size_um[None, :]
            seg_lengths = np.linalg.norm(np.diff(pts_um, axis=0), axis=1)
            graph_weighted.edges[u, v]['path_length_um'] = float(np.sum(seg_lengths))

        node_to_idx = {n: i for i, n in enumerate(nodes)}
        dist_matrix = np.full((len(nodes), len(nodes)), np.inf, dtype=float)
        for source in nodes:
            i = node_to_idx[source]
            lengths = nx.single_source_dijkstra_path_length(
                graph_weighted, source, weight='path_length_um'
            )
            for target, d in lengths.items():
                j = node_to_idx[target]
                dist_matrix[i, j] = float(d)
    else:
        dist_matrix = cdist(positions_um, positions_um)

    endpoint_mask = (node_type == 'sprout')
    branch_point_mask = (node_type == 'junction')

    if np.any(branch_point_mask):
        sub_dist = dist_matrix[np.ix_(branch_point_mask, branch_point_mask)].copy()
        np.fill_diagonal(sub_dist, np.inf)
        nearest = np.min(sub_dist, axis=1)
        nearest = nearest[np.isfinite(nearest)]
        if nearest.size > 0:
            summary['median_junction_dist_nearest_junction_um'] = safe_median(nearest)
            summary['p90_minus_p10_junction_dist_nearest_junction_um'] = safe_percentile_spread(nearest)

    if np.any(endpoint_mask):
        sub_dist = dist_matrix[np.ix_(endpoint_mask, endpoint_mask)].copy()
        np.fill_diagonal(sub_dist, np.inf)
        nearest = np.min(sub_dist, axis=1)
        nearest = nearest[np.isfinite(nearest)]
        if nearest.size > 0:
            summary['median_sprout_dist_nearest_endpoint_um'] = safe_median(nearest)
            summary['p90_minus_p10_sprout_dist_nearest_endpoint_um'] = safe_percentile_spread(nearest)

    return summary

In [4]:
def compute_internal_pore_headline_metrics(
    mask,
    voxel_size_um=(2.0, 2.0, 2.0),
    min_pore_area_um2=16.0,
    max_pore_area_fraction_of_slice=0.15,
    use_gpu_edt=True,
    ):
    voxel_size_um = np.asarray(voxel_size_um, dtype=float)
    _, y_um, x_um = voxel_size_um
    pixel_area_um2 = float(y_um * x_um)

    total_pore_area_um2 = 0.0
    total_filled_area_um2 = 0.0
    pore_areas_all = []
    pore_radii_all = []

    for z in range(mask.shape[0]):
        vessel_slice = mask[z].astype(bool)
        filled_slice = ndi.binary_fill_holes(vessel_slice)
        internal_pores = filled_slice & ~vessel_slice

        filled_area_um2 = float(np.count_nonzero(filled_slice)) * pixel_area_um2
        total_filled_area_um2 += filled_area_um2

        if not np.any(internal_pores):
            continue

        labeled, n_labels = ndi.label(internal_pores, structure=np.ones((3, 3), dtype=np.uint8))
        if n_labels == 0:
            continue

        area_counts = np.bincount(labeled.ravel(), minlength=n_labels + 1)[1:].astype(np.float64)
        area_um2_all = area_counts * pixel_area_um2

        slice_area_um2 = float(vessel_slice.size) * pixel_area_um2
        max_pore_area_um2 = float(max_pore_area_fraction_of_slice) * slice_area_um2

        label_ids = np.arange(1, n_labels + 1, dtype=np.int32)
        valid_mask = (area_um2_all >= min_pore_area_um2) & (area_um2_all <= max_pore_area_um2)
        if not np.any(valid_mask):
            continue

        valid_label_ids = label_ids[valid_mask]
        valid_areas = area_um2_all[valid_mask]

        if use_gpu_edt:
            dist_map_um = cp.asnumpy(
                ndi_gpu.distance_transform_edt(
                    cp.asarray(internal_pores, dtype=cp.uint8),
                    sampling=(float(y_um), float(x_um)),
                )
            )
        else:
            dist_map_um = edt(internal_pores, sampling=(y_um, x_um))

        valid_radii = np.asarray(
            ndi.maximum(dist_map_um, labels=labeled, index=valid_label_ids),
            dtype=float,
        )

        pore_areas_all.append(valid_areas)
        pore_radii_all.append(valid_radii)
        total_pore_area_um2 += float(np.sum(valid_areas))

    if len(pore_areas_all) == 0:
        return {
            'total_internal_pore_count': 0,
            'internal_pore_area_fraction_in_filled_vascular_area': 0.0,
            'median_internal_pore_area_um2': np.nan,
            'p90_minus_p10_internal_pore_area_um2': np.nan,
            'median_internal_pore_max_inscribed_radius_um': np.nan,
            'p90_minus_p10_internal_pore_max_inscribed_radius_um': np.nan,
        }

    all_areas = np.concatenate(pore_areas_all)
    all_radii = np.concatenate(pore_radii_all)

    return {
        'total_internal_pore_count': int(all_areas.size),
        'internal_pore_area_fraction_in_filled_vascular_area': float(total_pore_area_um2 / max(total_filled_area_um2, 1e-12)),
        'median_internal_pore_area_um2': float(np.median(all_areas)),
        'p90_minus_p10_internal_pore_area_um2': float(np.percentile(all_areas, 90) - np.percentile(all_areas, 10)),
        'median_internal_pore_max_inscribed_radius_um': float(np.median(all_radii)),
        'p90_minus_p10_internal_pore_max_inscribed_radius_um': float(np.percentile(all_radii, 90) - np.percentile(all_radii, 10)),
    }

def build_internal_pore_label_volumes(
    mask,
    voxel_size_um=(2.0, 2.0, 2.0),
    max_pore_area_fraction_of_slice=0.15,
    ):
    voxel_size_um = np.asarray(voxel_size_um, dtype=float)
    holes = np.zeros_like(mask, dtype=np.uint8)
    hole_labels_per_slice = np.zeros_like(mask, dtype=np.int32)
    hole_distance_per_slice_um = np.zeros_like(mask, dtype=np.float32)

    for z in range(mask.shape[0]):
        vessel_slice = mask[z].astype(bool)
        filled_slice = ndi.binary_fill_holes(vessel_slice)
        internal_pores = filled_slice & ~vessel_slice

        labeled, n_labels = ndi.label(internal_pores, structure=np.ones((3, 3), dtype=np.uint8))
        if n_labels == 0:
            continue

        area_counts = np.bincount(labeled.ravel(), minlength=n_labels + 1).astype(np.float64)
        max_pore_area_px = float(max_pore_area_fraction_of_slice) * float(vessel_slice.size)
        valid_label_mask = (area_counts > 0) & (area_counts <= max_pore_area_px)
        valid_label_mask[0] = False

        filtered_pores_slice = valid_label_mask[labeled]
        holes[z] = filtered_pores_slice.astype(np.uint8)

        relabeled_slice, _ = ndi.label(filtered_pores_slice, structure=np.ones((3, 3), dtype=np.uint8))
        hole_labels_per_slice[z] = relabeled_slice.astype(np.int32)

        if np.any(filtered_pores_slice):
            hole_distance_per_slice_um[z] = edt(
                filtered_pores_slice,
                sampling=tuple(voxel_size_um[1:]),
            ).astype(np.float32)

    return holes, hole_labels_per_slice, hole_distance_per_slice_um

In [5]:
def clean_and_analyse(vasculature_segmentation, to_display=True, junction_distance_mode='skeleton'):
    ##################### CLEAN SEGMENTATION #####################
    voxel_size_um = np.array([2.0, 2.0, 2.0], dtype=float)
    voxel_volume_um3 = float(np.prod(voxel_size_um))
    chunk_size = (vasculature_segmentation.shape[0], 512, 512)
    max_internal_pore_area_fraction_of_slice = 0.10
    binary_smoothed = cupy_chunk_processing(
        volume=vasculature_segmentation.astype(np.float32),
        processing_func=ndi_gpu.gaussian_filter,
        sigma=3,
        chunk_size=chunk_size,
    ) > 0.5
    clean_segmentation = cupy_chunk_processing(
        volume=binary_smoothed,
        processing_func=ndi_gpu.binary_fill_holes,
        chunk_size=chunk_size,
    ).astype(np.float32)
    binary_edt = cupy_chunk_processing(
        volume=clean_segmentation,
        processing_func=ndi_gpu.distance_transform_edt,
        sampling=tuple(voxel_size_um),
        chunk_size=chunk_size,
    )

    ##################### SKELETONISE AND GRAPH #####################
    dask_volume = da.from_array(clean_segmentation.astype(bool), chunks=chunk_size)
    skeleton = da.overlap.map_overlap(
        dask_volume,
        skeletonize_3d,
        depth=(2, 2, 2),
        boundary='none',
        dtype=bool,
    ).compute(scheduler='threads')
    graph = sknw.build_sknw(skeleton)
    area_image = compute_cross_sectional_areas(
        vasculature_segmentation,
        skeleton,
        binary_edt,
        voxel_size_um=voxel_size_um,
    )

    for n in list(graph.nodes()):
        graph.nodes[n]['pts'] = graph.nodes[n]['pts'][0]
        graph.nodes[n]['sprout'] = graph.degree(n) == 1

    pruned_graph = prune_graph(graph=graph, area_3d=area_image, edt_cutoff=0.20, length_cutoff=25)
    clean_graph = remove_mid_node(pruned_graph)
    clean_graph = collect_border_vicinity_edges(clean_graph, vasculature_segmentation.shape)
    isolated_nodes = [node for node in clean_graph.nodes() if clean_graph.degree[node] == 0]
    if isolated_nodes:
        clean_graph.remove_nodes_from(isolated_nodes)

    skeleton_from_graph = graph2image(clean_graph, vasculature_segmentation.shape).astype(np.uint8)
    for node in clean_graph.nodes():
        clean_graph.nodes[node]['sprout'] = clean_graph.degree(node) == 1

    ##################### METRICS #####################
    global_metrics = {}
    chip_volume_um3 = float(np.prod(vasculature_segmentation.shape)) * voxel_volume_um3
    vessel_volume_um3 = float(np.count_nonzero(clean_segmentation)) * voxel_volume_um3
    chip_characteristic_length_um = float(np.cbrt(max(chip_volume_um3, 1e-12)))
    chip_characteristic_area_um2 = chip_characteristic_length_um ** 2

    if clean_graph.number_of_edges() > 0:
        fd, lacunarity = fractal_dimension_and_lacunarity(skeleton_from_graph > 0)
        total_vessel_length_um = np.sum([
            np.linalg.norm(
                np.diff(clean_graph[u][v]['pts'].astype(float) * voxel_size_um[None, :], axis=0),
                axis=1,
            ).sum()
            for u, v in clean_graph.edges()
        ])
        branchpoints_count = sum(1 for u in clean_graph.nodes() if not clean_graph.nodes[u]['sprout'])
        sprouts_count = sum(
            1 for u, v in clean_graph.edges()
            if clean_graph.nodes[u]['sprout'] or clean_graph.nodes[v]['sprout']
        )
    else:
        fd, lacunarity = np.nan, np.nan
        total_vessel_length_um = 0.0
        branchpoints_count = 0
        sprouts_count = 0

    global_metrics['chip_volume_um3'] = chip_volume_um3
    global_metrics['vessel_volume_um3'] = vessel_volume_um3
    global_metrics['vessel_volume_fraction'] = safe_divide(vessel_volume_um3, chip_volume_um3)
    global_metrics['total_vessel_length_um'] = float(total_vessel_length_um)
    global_metrics['vessel_length_per_chip_volume_um_inverse2'] = safe_divide(total_vessel_length_um, chip_volume_um3)
    global_metrics['sprouts_per_vessel_length_um_inverse'] = safe_divide(sprouts_count, total_vessel_length_um)
    global_metrics['junctions_per_vessel_length_um_inverse'] = safe_divide(branchpoints_count, total_vessel_length_um)
    global_metrics['skeleton_fractal_dimension'] = fd
    global_metrics['skeleton_lacunarity'] = lacunarity
    global_metrics['median_sprout_and_branch_tortuosity'] = np.nan
    global_metrics['p90_minus_p10_sprout_and_branch_tortuosity'] = np.nan
    global_metrics['median_sprout_and_branch_median_cs_area_um2'] = np.nan
    global_metrics['p90_minus_p10_sprout_and_branch_median_cs_area_um2'] = np.nan
    global_metrics['median_junction_dist_nearest_junction_um'] = np.nan
    global_metrics['p90_minus_p10_junction_dist_nearest_junction_um'] = np.nan
    global_metrics['median_sprout_dist_nearest_endpoint_um'] = np.nan
    global_metrics['p90_minus_p10_sprout_dist_nearest_endpoint_um'] = np.nan

    if clean_graph.number_of_nodes() > 0 and clean_graph.number_of_edges() > 0:
        global_metrics.update(
            summarize_network_headline_metrics(
                clean_graph,
                area_image,
                voxel_size_um=voxel_size_um,
                distance_mode=junction_distance_mode,
            )
        )

    pore_global_metrics = compute_internal_pore_headline_metrics(
        clean_segmentation.astype(bool),
        voxel_size_um=voxel_size_um,
        min_pore_area_um2=16.0,
        max_pore_area_fraction_of_slice=max_internal_pore_area_fraction_of_slice,
        use_gpu_edt=True,
    )
    global_metrics.update(pore_global_metrics)

    ##################### DISPLAY INPUTS #####################
    holes = np.zeros_like(clean_segmentation, dtype=np.uint8)
    hole_labels_per_slice = np.zeros_like(clean_segmentation, dtype=np.int32)
    hole_distance_per_slice_um = np.zeros_like(clean_segmentation, dtype=np.float32)
    max_label = 0
    if to_display:
        holes, hole_labels_per_slice, hole_distance_per_slice_um = build_internal_pore_label_volumes(
            clean_segmentation.astype(bool),
            voxel_size_um=voxel_size_um,
            max_pore_area_fraction_of_slice=max_internal_pore_area_fraction_of_slice,
        )
        max_label = int(hole_labels_per_slice.max())

    ##################### SHAPE-INVARIANT NORMALIZATION #####################
    global_metrics['chip_characteristic_length_um'] = chip_characteristic_length_um
    global_metrics['total_vessel_length_per_chip_characteristic_length'] = safe_divide(
        global_metrics.get('total_vessel_length_um', np.nan),
        chip_characteristic_length_um,
    )
    global_metrics['sprouts_per_chip_volume_um_inverse3'] = safe_divide(sprouts_count, chip_volume_um3)
    global_metrics['junctions_per_chip_volume_um_inverse3'] = safe_divide(branchpoints_count, chip_volume_um3)
    global_metrics['median_junction_dist_nearest_junction_per_characteristic_length'] = safe_divide(
        global_metrics.get('median_junction_dist_nearest_junction_um', np.nan),
        chip_characteristic_length_um,
    )
    global_metrics['p90_minus_p10_junction_dist_nearest_junction_per_characteristic_length'] = safe_divide(
        global_metrics.get('p90_minus_p10_junction_dist_nearest_junction_um', np.nan),
        chip_characteristic_length_um,
    )
    global_metrics['median_sprout_dist_nearest_endpoint_per_characteristic_length'] = safe_divide(
        global_metrics.get('median_sprout_dist_nearest_endpoint_um', np.nan),
        chip_characteristic_length_um,
    )
    global_metrics['p90_minus_p10_sprout_dist_nearest_endpoint_per_characteristic_length'] = safe_divide(
        global_metrics.get('p90_minus_p10_sprout_dist_nearest_endpoint_um', np.nan),
        chip_characteristic_length_um,
    )
    global_metrics['median_cs_area_over_characteristic_area'] = safe_divide(
        global_metrics.get('median_sprout_and_branch_median_cs_area_um2', np.nan),
        chip_characteristic_area_um2,
    )
    global_metrics['p90_minus_p10_cs_area_over_characteristic_area'] = safe_divide(
        global_metrics.get('p90_minus_p10_sprout_and_branch_median_cs_area_um2', np.nan),
        chip_characteristic_area_um2,
    )
    global_metrics['total_internal_pore_density_per_vessel_volume_um_inverse3'] = safe_divide(
        global_metrics.get('total_internal_pore_count', np.nan),
        vessel_volume_um3,
    )

    global_metrics_df = pd.DataFrame([global_metrics])
    return {
        'global_metrics': global_metrics,
        'global_metrics_df': global_metrics_df,
        'voxel_size_um': voxel_size_um,
        'chunk_size': chunk_size,
        'clean_segmentation': clean_segmentation,
        'binary_edt': binary_edt,
        'skeleton': skeleton,
        'graph': graph,
        'area_image': area_image,
        'pruned_graph': pruned_graph,
        'clean_graph': clean_graph,
        'skeleton_from_graph': skeleton_from_graph,
        'holes': holes,
        'hole_labels_per_slice': hole_labels_per_slice,
        'hole_distance_per_slice_um': hole_distance_per_slice_um,
        'max_label': max_label,
    }

In [6]:
vasculature_segmentation= np.load(r"C:\Users\taylorhearn\git_repos\vascumap\working_outputs_double_crop\20250619_Vascumap_Dev25_11_Daisy10_20250619_Vascumap_Dev25_11_Daisy10_vessel_mask_iso_354_cropped.npy")
vasculature_segmentation = vasculature_segmentation[:, 50:2321, 130:1449]


analysis_outputs = clean_and_analyse(vasculature_segmentation, to_display=True)
global_metrics = analysis_outputs['global_metrics']
global_metrics_df = analysis_outputs['global_metrics_df']
voxel_size_um = analysis_outputs['voxel_size_um']
chunk_size = analysis_outputs['chunk_size']
clean_segmentation = analysis_outputs['clean_segmentation']
binary_edt = analysis_outputs['binary_edt']
skeleton = analysis_outputs['skeleton']
graph = analysis_outputs['graph']
area_image = analysis_outputs['area_image']
pruned_graph = analysis_outputs['pruned_graph']
clean_graph = analysis_outputs['clean_graph']
skeleton_from_graph = analysis_outputs['skeleton_from_graph']
holes = analysis_outputs['holes']
hole_labels_per_slice = analysis_outputs['hole_labels_per_slice']
hole_distance_per_slice_um = analysis_outputs['hole_distance_per_slice_um']
max_label = analysis_outputs['max_label']

Processing chunks: 100%|██████████| 1/1 [00:01<00:00,  1.49s/it]
c:\Users\taylorhearn\AppData\Local\miniconda3\envs\clean_vascumap\Lib\site-packages\dask\array\overlap.py:667: FutureWarning: The use of map_overlap(array, func, **kwargs) is deprecated since dask 2.17.0 and will be an error in a future release. To silence this warning, use the syntax map_overlap(func, array0,[ array1, ...,] **kwargs) instead.
  warnings.warn(


In [8]:
# Single final visualization display
viewer = napari.Viewer(ndisplay=3)
from skimage.measure import label
holes_labels = label(holes)

viewer.add_labels(clean_segmentation.astype(np.int32), colormap = custom_napari_palette(1,"dodgerblue", hue_span=0, sat_span=0, light_span=0))
viewer.add_labels(holes_labels.astype(np.int32), colormap = custom_napari_palette(len(np.unique(holes_labels)),"red", hue_span=0.1, sat_span=0.1, light_span=0.1))


pore_labels_layer = viewer.add_labels(hole_labels_per_slice.astype(np.int32), name='pore_labels', colormap = custom_napari_palette(max_label,"limegreen", hue_span=0.2, sat_span=0.5, light_span=0.5))


viewer.add_image(hole_distance_per_slice_um.astype(np.float32), name='hole_distance_per_slice_um', colormap='greens')
# _add_colored_binary_labels(skeleton, 'raw_skeleton', 'deeppink', opacity=0.70, alpha=0.95)
# _add_colored_binary_labels(skeleton_from_graph, 'pruned_graph_skeleton', 'darkviolet', opacity=0.95, alpha=1.0)

# Graph nodes (sprout/junction)
node_ids = list(clean_graph.nodes())
node_pts = np.array([clean_graph.nodes[n]['pts'] for n in node_ids], dtype=float)
if len(node_pts) > 0:
    is_sprout = np.array([bool(clean_graph.nodes[n].get('sprout', False)) for n in node_ids], dtype=bool)
    sprout_pts = node_pts[is_sprout]
    junction_pts = node_pts[~is_sprout]

    if len(junction_pts) > 0:
        viewer.add_points(junction_pts, name='graph_nodes_junction')
    if len(sprout_pts) > 0:
        viewer.add_points(sprout_pts, name='graph_nodes_sprout')

c:\Users\taylorhearn\AppData\Local\miniconda3\envs\clean_vascumap\Lib\site-packages\napari\utils\colormaps\colormap.py:455: UserWarning: color_dict did not provide a default color. Missing keys will be transparent. To provide a default color, use the key `None`, or provide a defaultdict instance.
  warn(
c:\Users\taylorhearn\AppData\Local\miniconda3\envs\clean_vascumap\Lib\site-packages\napari\utils\colormaps\colormap.py:455: UserWarning: color_dict did not provide a default color. Missing keys will be transparent. To provide a default color, use the key `None`, or provide a defaultdict instance.
  warn(
c:\Users\taylorhearn\AppData\Local\miniconda3\envs\clean_vascumap\Lib\site-packages\napari\utils\colormaps\colormap.py:455: UserWarning: color_dict did not provide a default color. Missing keys will be transparent. To provide a default color, use the key `None`, or provide a defaultdict instance.
  warn(
